In [ ]:
# ==== Stacked Bar Plot (Default Colors & Overall P-value) ====

suppressWarnings(suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(ggplot2)
  library(vegan)
}))

input_path <- "CIBERSORT_GroupComparison/csv/fractions_long_with_group.csv"
long_df <- read.csv(input_path, check.names = FALSE)

long_df <- long_df %>%
  mutate(
    Group = factor(Group, levels = c("Low", "High")),
    Fraction = as.numeric(Fraction)
  )


wide_df <- long_df %>%
  select(Sample, Group, Cell, Fraction) %>%
  pivot_wider(names_from = Cell, values_from = Fraction, values_fill = 0)

cell_matrix <- wide_df %>% select(-Sample, -Group)

set.seed(123)
perm_test <- adonis2(cell_matrix ~ Group, data = wide_df, method = "bray", permutations = 9999)
overall_p <- perm_test$`Pr(>F)`[1]

if (!is.na(overall_p) && overall_p < 0.001) {
  pval_label <- "Overall immune  P-value < 0.001"
} else {
  pval_label <- sprintf("Overall immune  P-value = %.4f", overall_p)
}


cibersort_order <- c(
  "B cells naive", "B cells memory", "Plasma cells",
  "T cells CD8", "T cells CD4 naive", "T cells CD4 memory resting",
  "T cells CD4 memory activated", "T cells CD4 memory activited", 
  "T cells follicular helper", "T cells regulatory (Tregs)", "T cells regulatory",
  "T cells gamma delta", "NK cells resting", "NK cells activated",
  "Monocytes", "Macrophages M0", "Macrophages M1", "Macrophages M2",
  "Dendritic cells resting", "Dendritic cells activated",
  "Mast cells resting", "Mast cells activated",
  "Eosinophils", "Neutrophils"
)

actual_cells <- unique(long_df$Cell)
final_cell_order <- intersect(cibersort_order, actual_cells)
final_cell_order <- c(final_cell_order, setdiff(actual_cells, final_cell_order)) 
long_df$Cell <- factor(long_df$Cell, levels = final_cell_order)


sort_samples <- function(sub_df) {
  if(nrow(sub_df) <= 1) return(sub_df$Sample)
  mat <- as.matrix(sub_df %>% select(-Sample, -Group))
  rownames(mat) <- sub_df$Sample
  hc <- hclust(dist(mat), method = "ward.D2")
  return(sub_df$Sample[hc$order])
}

low_order <- sort_samples(wide_df %>% filter(Group == "Low"))
high_order <- sort_samples(wide_df %>% filter(Group == "High"))
long_df$Sample <- factor(long_df$Sample, levels = c(low_order, high_order))


p_final_auto <- ggplot(long_df, aes(x = Sample, y = Fraction, fill = Cell)) +
  geom_bar(stat = "identity", width = 1.05, color = NA) +
  facet_grid(~ Group, scales = "free_x", space = "free_x") +
  labs(
    x = "Samples", 
    y = "Relative Percentage", 
    fill = "Cell Type",
    subtitle = pval_label
  ) +
  scale_y_continuous(expand = c(0, 0), labels = scales::percent_format(scale = 100)) +
  theme_minimal(base_size = 14) +
  theme(
    plot.subtitle = element_text(size = 13, face = "bold", color = "#333333", hjust = 0.5, margin = margin(b = 12)),
    axis.text.x = element_blank(),
    axis.ticks.x = element_blank(),
    axis.text.y = element_text(color = "black", size = 12),
    strip.text = element_text(size = 16, face = "bold", margin = margin(b = 8, t = 8)),
    strip.background = element_rect(fill = "#f0f0f0", color = "black", linewidth = 0.5),
    legend.position = "right",
    legend.key.size = unit(0.45, "cm"),
    legend.text = element_text(size = 10),
    legend.title = element_text(size = 12, face = "bold"),
    panel.grid = element_blank(),
    panel.spacing = unit(0.2, "lines"),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5)
  ) +
  guides(fill = guide_legend(ncol = 1))

print(p_final_auto)
ggsave("CIBERSORT_StackedBar_AutoColors_Pval.png", p_final_auto, width = 16, height = 7.5, dpi = 300)

In [ ]:
# ==== Professional CIBERSORT Boxplot ====

suppressWarnings(suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(ggpubr)
}))

input_path <- "CIBERSORT_GroupComparison/csv/fractions_long_with_group.csv"
long_df <- read.csv(input_path, check.names = FALSE)

long_df <- long_df %>%
  mutate(
    Group = factor(Group, levels = c("Low", "High")),
    Fraction = as.numeric(Fraction)
  )

cell_order <- long_df %>%
  group_by(Cell) %>%
  summarise(med = median(Fraction, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(med)) %>% 
  pull(Cell)

long_df$Cell <- factor(long_df$Cell, levels = cell_order)

my_colors <- c("Low" = "#2166AC", "High" = "#B2182B")

max_val <- max(long_df$Fraction, na.rm = TRUE)

p_box <- ggplot(long_df, aes(x = Cell, y = Fraction, fill = Group)) +
  geom_boxplot(
    width = 0.7,                   
    position = position_dodge(0.8), 
    outlier.shape = 16,
    outlier.size = 0.7,
    outlier.alpha = 0.6,          
    color = "black",
    lwd = 0.45,
    alpha = 0.9
  ) +
  scale_fill_manual(values = my_colors) +
  labs(x = NULL, y = "Proportion") +
  theme_classic(base_size = 14) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black", size = 11),
    axis.text.y = element_text(color = "black", size = 12),
    axis.title.y = element_text(face = "bold", margin = margin(r = 12)),
    legend.position = "top",
    legend.title = element_blank(),
    legend.text = element_text(size = 12),
    plot.margin = margin(t = 15, r = 10, b = 10, l = 10)
  ) +
  stat_compare_means(
    aes(group = Group),
    label = "p.signif",
    method = "wilcox.test",
    hide.ns = FALSE,
    label.y = max_val + 0.04,     
    size = 4.5,                   
    symnum.args = list(
      cutpoints = c(0, 0.001, 0.01, 0.05, 1),
      symbols = c("***", "**", "*", "ns")
    )
  ) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.12)))

print(p_box)
ggsave("CIBERSORT_Publication_Ready.png", p_box, width = 16, height = 7, dpi = 300)